## Benchmark of DES techniques 

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, BaggingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB, ComplementNB
from sklearn.neighbors import KNeighborsClassifier

# Optional: fetch datasets
from pmlb import fetch_data
from sklearn.datasets import fetch_openml
import time
import tracemalloc
from sklearn.ensemble import VotingClassifier, StackingClassifier

from tqdm import tqdm 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- Evaluation seeds ---
seeds = [0, 42, 123, 2021, 7]

# --- Helper function for dataset fetching ---
def load_dataset(name):
    X, y = fetch_data(datasets[name], return_X_y=True)
    y = y.astype(int)
    return X, y

### Setup  

In [ ]:
classifiers = {
    "RF": RandomForestClassifier(random_state=42),
    "GB": GradientBoostingClassifier(random_state=42),
    "ET": ExtraTreesClassifier(random_state=42),
    "XGB": HistGradientBoostingClassifier(random_state=42),
    "CatBoost": HistGradientBoostingClassifier(random_state=42),  
    "LGBM": HistGradientBoostingClassifier(random_state=42),      
    "Bagging": BaggingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "SVC": SVC(probability=True, random_state=42),  
    "KNN": KNeighborsClassifier(),  
    "LR": LogisticRegression(max_iter=1000, random_state=42),
    "DT": DecisionTreeClassifier(random_state=42),
    "MLPClassifier": MLPClassifier(max_iter=500, random_state=42),
}

In [ ]:
datasets = {
    "Wine Quality Red": "wine_quality_red",
    "Wine Quality White": "wine_quality_white",
    "Led 24": "led24",
    "Led 7": "led7",
    "Yeast": "yeast",
    "ecoli": "ecoli",
    "vehicle": "vehicle",
    "Satimage": "satimage",
    "adult": "adult",
    "glass": "glass2",
    "wine_recognition": "wine_recognition",
    "collins": "collins",
    "soybean": "soybean",
    "penguins": "penguins",
    "heart_statlog": "saheart",
    "ionosphere": "ionosphere",
    "sonar": "sonar",
    "phoneme": "phoneme",
    "texture": "texture",
    "car": "car_evaluation",
    "breast": "breast_cancer",
    "dna": "dna",
    "shuttle": "shuttle",
    "magic": "magic",
    "fars": "fars",
}

### Add Static Ensembles

In [ ]:
from sklearn.base import clone 

extra_models = {
    "Voting_Hard": VotingClassifier(
        estimators=[(name, clone(clf)) for name, clf in classifiers.items()],
        voting="hard"
    ),
    "Voting_Soft": VotingClassifier(
        estimators=[(name, clone(clf)) for name, clf in classifiers.items()],
        voting="soft"
    ),
    "Stacking": StackingClassifier(
        estimators=[(name, clone(clf)) for name, clf in classifiers.items()],
        final_estimator=LogisticRegression(max_iter=1000),
        passthrough=False
    )
}

### Run

In [ ]:
from deslib.des.des_clustering import DESClustering
from deslib.des.des_knn import DESKNN
from deslib.des.knora_e import KNORAE
from deslib.des.knora_u import KNORAU
from deslib.des.probabilistic.rrc import RRC
from deslib.des.probabilistic.deskl import DESKL
from deslib.des.des_p import DESP
from deslib.des.knop import KNOP
from deslib.des.meta_des import METADES
from deslib.des.des_mi import DESMI

from des_models.gnn_des import GNNDES
from des_models.fh_des import DESFH
from des_models.im_def import imDEF
from des_models.des_as import DES_AS
from des_models.inca_des import IncA_DES


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score
from tqdm import tqdm
from sklearn.base import clone

# --- Main loop ---
dataset_names = list(datasets.keys())
ens_models_names = ["KNOP", "DESClustering", "DESKNN", "KNORAE", "KNORAU", "FIRE-DES", 
                    "DESMI", "RRC", "DESKL", "DESP", "METADES", "GNNDES", "DESFH", "imDEF", "DES_AS", "IncA_DES"]

# Initialize results dictionary
results_per_dataset = {ens: {ds: [] for ds in dataset_names} for ens in ens_models_names} 

for seed in seeds:
    print(f"\n--- Seed {seed} ---")
    for dataset_name in dataset_names:
        print(f"Dataset: {dataset_name}")
        X, y = load_dataset(dataset_name)
        
        # Split: Train 60%, Temp 40% (for DSEL + Test)
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.4, stratify=y, random_state=seed
        )
        # Split temp into DSEL 20%, Test 20%
        X_dsel, X_test, y_dsel, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=seed
        )
        
        # Scaling (MinMax)
        scaler_mm = MinMaxScaler()
        X_train_mm = scaler_mm.fit_transform(X_train)
        X_dsel_mm = scaler_mm.transform(X_dsel)
        X_test_mm = scaler_mm.transform(X_test)
        
        # -------------------------
        # Train pool of base classifiers
        # -------------------------
        trained_pool = []
        tracemalloc.start() 
        start_base_train = time.perf_counter() 
        for clf_name, clf in tqdm(classifiers.items(), desc="Training base classifiers"):
            clf_clone = clone(clf)
            clf_clone.fit(X_train_mm, y_train)
            trained_pool.append(clf_clone)
        end_base_train = time.perf_counter()
        base_train_time = end_base_train - start_base_train
        current_base, peak_base = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        base_memory_mb = peak_base / 1024**2

        # -------------------------
        # Initialize DES / Ensemble models
        # -------------------------
        ens_models = {
            "KNOP": KNOP(trained_pool, k=7),
            "DESClustering": DESClustering(trained_pool),
            "DESKNN": DESKNN(trained_pool, k=7),
            "KNORAE": KNORAE(trained_pool, k=7),
            "KNORAU": KNORAU(trained_pool, k=7),
            "FIRE-DES": KNORAU(trained_pool, k=7, DFP=True),
            "DESMI": DESMI(trained_pool, k=7),
            "RRC": RRC(trained_pool, k=7),
            "DESKL": DESKL(trained_pool, k=7),
            "DESP": DESP(trained_pool, k=7),
            "METADES": METADES(trained_pool, k=7),
            
            "GNNDES": GNNDES(trained_pool),
            "DESFH": DESFH(trained_pool),
            "imDEF": imDEF(trained_pool),
            "DES_AS": DES_AS(trained_pool),
            "IncA_DES": IncA_DES(base_classifier=clone(trained_pool[0]), k=7)
        }

        all_models = {**ens_models, **extra_models}

        for name, model in tqdm(all_models.items(), desc="Evaluating All Ensembles"):

            tracemalloc.start()

            # ---------------------
            # TRAINING TIME
            # ---------------------
            start_train = time.perf_counter()

            if name in ["Voting_Hard", "Voting_Soft", "Stacking"]:
                model.fit(X_train_mm, y_train)
            else:
                model.fit(X_dsel_mm, y_dsel)

            end_train = time.perf_counter()
            train_time = end_train - start_train

            # ---------------------
            # INFERENCE TIME
            # ---------------------
            start_test = time.perf_counter()

            try:
                y_pred = model.predict(X_test_mm)
            except:
                y_pred = np.array([model.predict(np.array([x]))[0] for x in X_test_mm])

            end_test = time.perf_counter()
            test_time = end_test - start_test

            # ---------------------
            # MEMORY
            # ---------------------
            current, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            memory_mb = peak / 1024**2

            # ---------------------
            # METRICS
            # ---------------------
            f1 = f1_score(y_test, y_pred, average='weighted')

            results_per_dataset.setdefault(name, {}) \
                       .setdefault(dataset_name, []) \
                       .append({
                        "f1": f1,
                        "train_time": train_time,
                        "test_time": test_time,
                        "memory_mb": memory_mb
                    })

### Summary

In [ ]:
import pandas as pd
import numpy as np

rows = []

for method, datasets_dict in results_per_dataset.items():
    for dataset, scores in datasets_dict.items():

        if len(scores) == 0:
            continue

        # ---- Extract metrics ----
        f1s = [s["f1"] for s in scores]

        train_times = [s["train_time"] for s in scores]
        test_times = [s["test_time"] for s in scores]

        memories = [s["memory_mb"] for s in scores]

        # ---- Compute statistics ----
        f1_mean = np.mean(f1s) * 100
        f1_std = np.std(f1s, ddof=1) * 100

        train_mean = np.mean(train_times)
        train_std = np.std(train_times, ddof=1)

        test_mean = np.mean(test_times)
        test_std = np.std(test_times, ddof=1)

        mem_mean = np.mean(memories)
        mem_std = np.std(memories, ddof=1)

        rows.append({
            "Method": method,
            "Dataset": dataset,

            "F1 (%)": f"{f1_mean:.3f}±{f1_std:.3f}",

            "Total Train Time (s)": f"{train_mean:.3f}±{train_std:.4f}",
            "Test Time (s)": f"{test_mean:.3f}±{test_std:.3f}",

            "Total Memory (MB)": f"{mem_mean:.3f}±{mem_std:.3f}",
        })

df_results = pd.DataFrame(rows)

In [ ]:
df_results